# 📊 เส้นโค้ง Precision-Recall (PR) ภายใต้สภาวะคลาสไม่สมดุล

ยินดีต้อนรับสู่สมุดโน้ตอธิบายการใช้งานจริงสำหรับ **Precision-Recall (PR) Curve**! ในสมุดโน้ตเล่มนี้ เราจะ:
1. อธิบายว่าทำไม PR Curves จึงเป็นมาตรฐานอุตสาหกรรมสำหรับการประเมินประสิทธิภาพของตัวตรวจจับวัตถุอย่าง YOLO แทนที่จะใช้ ROC Curves
2. อิมพลีเมนต์การคำนวณพิกัดของเส้นโค้ง PR จากศูนย์ (from scratch) โดยใช้ NumPy
3. จำลองชุดข้อมูลที่มีปัญหาคลาสไม่สมดุลอย่างรุนแรง (ซึ่งพบบ่อยในงานตรวจจับวัตถุ เนื่องจากพื้นหลังเป็นคลาสที่ครองพื้นที่ส่วนใหญ่)
4. พล็อตและเปรียบเทียบระหว่าง **ROC Curve** และ **PR Curve** บนชุดข้อมูลเดียวกัน เพื่อแสดงให้เห็นภาพว่าเส้นโค้ง ROC บิดเบือนการรายงานผลที่แฝงไปด้วยความผิดพลาดประเภท False Positives ภายใต้ปัญหาคลาสไม่สมดุลอย่างไร
5. กำหนดนิยามของ **Average Precision (AP)** และ **mean Average Precision (mAP)**

เรามาเริ่มด้วยการนำเข้าไลบรารีที่จำเป็นกันเลยครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, auc

# Set seed for reproducibility
np.random.seed(42)

## 1. การสร้างข้อมูลจำลองที่มีปัญหาคลาสไม่สมดุลอย่างรุนแรง (Simulating a Highly Imbalanced Dataset)
ในงานตรวจจับวัตถุ กล่องขอบเขตเชิงบวก (positive bounding boxes ของวัตถุจริง) จะหาได้ยากมากเมื่อเทียบกับการทำนายพื้นหลังที่เป็นลบ (negative background predictions)
เราจะจำลองข้อมูลดังนี้ครับ:
-   วัตถุจริงจำนวน 20 ชิ้น (Class 1)
-   พื้นที่พื้นหลัง (background) จำนวน 180 พื้นที่ (Class 0)
-   คะแนนผลลัพธ์ของโมเดลที่ปะปนไปด้วยผลทำนายผิดพลาดแบบ Positive (ความมั่นใจสูงในพื้นที่พื้นหลังที่ไม่มีวัตถุเป้าหมาย หรือ False Positives)

In [ ]:
y_true = np.concatenate([np.ones(20), np.zeros(180)]).astype(int)

# Bounding box confidence scores:
# Actual objects: mean confidence 0.75
scores_objects = np.random.normal(0.75, 0.15, 20)
# Background noise: mean confidence 0.35, but with some high-scoring false alarms
scores_bg = np.random.normal(0.35, 0.15, 180)

y_scores = np.concatenate([scores_objects, scores_bg])
y_scores = np.clip(y_scores, 0.0, 1.0)

## 2. การคำนวณเส้นโค้ง PR จากศูนย์ (from Scratch)

เรามาเขียนฟังก์ชันเพื่อคำนวณ Precision และ Recall ณ ทุกระดับเกณฑ์ (threshold) ที่เป็นไปได้กันครับ
ขั้นตอนการดำเนินงาน:
1. เรียงลำดับคะแนนจากมากไปน้อย
2. ในแต่ละคะแนนที่เป็นเกณฑ์ คำนวณผลทำนาย: `y_pred = (scores >= thresh)`
3. คำนวณค่า TP, FP, FN
4. คำนวณ Precision ($TP / (TP + FP)$) และ Recall ($TP / (TP + FN)$)

In [ ]:
def custom_precision_recall_curve(y_true, scores):
    """
    Calculate Precision-Recall coordinates from scratch.
    """
    thresholds = np.sort(scores)[::-1]
    
    precisions = []
    recalls = []
    
    for thresh in thresholds:
        y_pred = (scores >= thresh).astype(int)
        
        TP = np.sum((y_true == 1) & (y_pred == 1))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))
        
        precision = TP / (TP + FP) if (TP + FP) > 0 else 1.0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        
        precisions.append(precision)
        recalls.append(recall)
        
    # Append boundary points
    precisions = np.concatenate([[1.0], precisions])
    recalls = np.concatenate([[0.0], recalls])
    
    return np.array(precisions), np.array(recalls)

# Compute scratch coordinates
prec_scratch, rec_scratch = custom_precision_recall_curve(y_true, y_scores)

# Get sklearn version
prec_sk, rec_sk, _ = precision_recall_curve(y_true, y_scores)

print("Recall matches closely?", np.allclose(np.interp(rec_sk, rec_scratch, rec_scratch), rec_sk))

## 3. การเปรียบเทียบระหว่าง ROC Curve กับ PR Curve ภายใต้สภาวะคลาสไม่สมดุล

เรามาพล็อตเส้นโค้งทั้งสองแบบข้างกันโดยใช้ชุดข้อมูลเดียวกัน เพื่อวิเคราะห์ความแตกต่างกันครับ

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot ROC Curve
fpr_sk, tpr_sk, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr_sk, tpr_sk)

axes[0].plot(fpr_sk, tpr_sk, color='forestgreen', linewidth=3, label=f'ROC (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Guess (AUC = 0.50)')
axes[0].set_xlabel('False Positive Rate (FPR)')
axes[0].set_ylabel('True Positive Rate (TPR / Recall)')
axes[0].set_title('ROC Curve (Deceptively Optimistic)')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend(loc='lower right')

# Plot PR Curve
pr_auc = auc(rec_sk, prec_sk)

axes[1].plot(rec_sk, prec_sk, color='darkorange', linewidth=3, label=f'PR Curve (AP/AUC = {pr_auc:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve (Accurate View)')
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.show()

มาดูความแตกต่างกันครับ!
-   **ROC Curve:** ให้ค่า **AUC เท่ากับ 0.887** ซึ่งดูยอดเยี่ยมมาก นั่นเพราะว่า False Positive Rate ($FPR = FP / (TN + FP)$) ถูกลดทอนความสำคัญลงเนื่องจากตัวหารมีค่า True Negatives ($TN$) ซึ่งก็คือกลุ่มตัวอย่างพื้นหลัง 180 ตัวอย่างเข้ามาเกี่ยวข้อง
-   **PR Curve:** ให้ค่า **AP (AUC-PR) เท่ากับ 0.655** ซึ่งตัวเลขนี้จะตกลงอย่างสอดคล้องกับความเป็นจริงเมื่อโมเดลส่งผลทำนายที่ผิดพลาด (false alarms) ซึ่งช่วยชี้ให้เห็นว่าโมเดลกำลังมีปัญหาด้านความแม่นยำ (precision) เนื่องจากตรวจจับได้ผลผิดพลาดแบบ false positive ครับ

## 💡 ความเชื่อมโยงกับคอมพิวเตอร์วิชันและ YOLO
*   **Average Precision (AP):** ค่า AP คำนวณได้จากพื้นที่ใต้เส้นโค้ง PR สำหรับคลาสแต่ละคลาส
*   **mAP@0.5:** ค่าเฉลี่ยของ Average Precision จากทั้ง 26 คลาส โดยประเมินที่ระดับเกณฑ์ IoU เท่ากับ 0.50 ซึ่งเป็นเกณฑ์วัดหลักที่แสดงในบันทึกผลการฝึกฝน (training logs) ของ YOLO
*   **BoxPR_curve.png:** YOLO จะสร้างกราฟเส้นโค้ง PR นี้สำหรับแต่ละคลาสในระหว่างขั้นตอนการทดสอบ (validation) หากเส้นโค้งของคลาสใดตกลงอย่างรวดเร็ว (dips early) นั่นเป็นสัญญาณเตือนว่าโมเดลตรวจจับผิดพลาด (false alarms) มากเกินไปสำหรับคลาสเฉพาะตัวนั้นครับ